In [5]:
%load_ext autoreload
%autoreload 2


import uuid

from typing import Union, Any, Dict, List

from sqlalchemy import insert, select, and_
from sqlalchemy.engine import Engine
from sqlalchemy.orm import sessionmaker
from sqlalchemy.orm.session import Session

from app.models.base import Base

mt = Base.metadata


from app.models.user import User, Resume, UserQueryPreference
from app.models.job import JobItem

from app.models.user import User, Resume, UserQueryPreference

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
engine = create_engine("sqlite:///test.db")

In [7]:
mt.create_all(engine, checkfirst=True)

In [8]:
from contextlib import contextmanager


@contextmanager
def session_scope(sessionmaker):
    """Provide a transactional scope around a series of operations."""
    session = sessionmaker()
    try:
        yield session
        session.commit()
    except:
        session.rollback()
        raise
    finally:
        session.close()

In [ ]:
from app.models.constant import JobSource

with session_scope(sessionmaker(bind=engine)) as session:
    jb_item = session.scalars(
        select(JobItem).where(JobItem.source == JobSource.ZHILIAN)
    ).first()

    print(jb_item)

MultipleResultsFound: Multiple rows were found when exactly one was required

In [4]:
from sqlalchemy.engine import Connection

[Doc on scope of `Session`](https://docs.sqlalchemy.org/en/20/orm/session_basics.html#when-do-i-construct-a-session-when-do-i-commit-it-and-when-do-i-close-it)

Keep the lifecycle of the session (and usually the transaction) separate and external. The example below illustrates how this might look, and additionally makes use of a Python context manager (i.e. the with: keyword) in order to manage the scope of the Session and its transaction automatically:

```python
### this is a **better** (but not the only) way to do it ###


class ThingOne:
    def go(self, session):
        session.execute(update(FooBar).values(x=5))


class ThingTwo:
    def go(self, session):
        session.execute(update(Widget).values(q=18))


def run_my_program():
    with Session() as session:
        with session.begin():
            ThingOne().go(session)
            ThingTwo().go(session)
```

In [5]:
import uuid

from typing import Union, Any, Dict, List

from sqlalchemy import insert, select
from sqlalchemy.engine import Engine
from sqlalchemy.orm.session import Session

In [10]:
class DBController:
    def __init__(self, engine: Engine):
        self.session_maker = sessionmaker(bind=engine)

    def insert_user(session: Session, user: Union[User, List[User]]):
        if isinstance(user, List):
            session.add_all(user)
        elif isinstance(user, User):
            session.add(user)
        else:
            raise TypeError(
                f"`user` parameter only supports a `User` or `List[User]` instance, but a type `{type(user)}` is passed in."
            )

    def get_user_by_uuid(session: Session, user_id: uuid):
        return session.scalars(select(User).where(User.id == user_id)).one()

    def upload_resume(session: Session, resume: Resume):
        session.add(resume, Resume)

FacadeDict({})

In [ ]:
with session_scope(sessionmaker(bind=engine)) as session:
    # 逐条增加新记录
    insert_user = User(id=uuid.uuid1(), username="a", password="123456")
    session.add(insert_user)

    insert_user = User(id=uuid.uuid1(), username="b", password="123")
    session.add(insert_user)

    # 插入多条记录
    session.add_all(
        [
            User(id=uuid.uuid1(), username="c", password="123123"),
            User(id=uuid.uuid1(), username="d", password="3456576"),
        ]
    )

    u = session.scalars(select(User).where(User.username == "c")).all()

    print(u)

In [13]:
with session_scope(sessionmaker(bind=engine)) as session:
    u = session.scalars(select(User).where(User.username == "c")).one()

    print(u.id, u.username, u.password)

8c8b4d3d-6c4e-11f0-a017-d0577e711c12 c 123123


In [ ]:
from sqlalchemy import text

statement = """
SELECT * FROM UserDB
"""

results = session.execute(text(statement))

In [ ]:
results.all()

In [ ]:
for record in results:
    print(record)

In [ ]:
type(results)